# Lean variance analysis for `merged.csv`

This notebook is intentionally small: load `merged.csv`, apply the requested preprocessing stub, run train/test-aware statistical tests, plot raw/FDR-corrected p-value distributions, and plot variance explained.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

plt.rcParams["svg.fonttype"] = "none"

DATA_PATH = Path("~/Desktop/plm_paper/merged.csv").expanduser()
OUT_DIR = Path("~/Desktop/plm_paper/refined_figures/supp_figure_var_analysis").expanduser()

# Set this to a CSV/TSV/parquet file to read a precomputed variance-explained table.
# Expected columns: dataset, feature or var, factor, var_explained.
VAR_EXPLAINED_PATH = None
CALCULATE_VARIANCE_EXPLAINED_IF_MISSING = True

SAVE_TABLES = True
SAVE_FIGURES = True

FEATURE_COLUMNS = ["roc", "top_100_pct", "correlation"]
MIN_GROUPS = 3
MIN_POINTS_PER_GROUP = 2


## Load and preprocess

Keep this function as the single preprocessing stub for `merged.csv` before analysis.


In [ ]:
def preprocess_merged_csv(df):
    """Preprocess merged.csv before any analysis."""
    df = df.copy()

    # Requested filtering stub.
    df = df[~df["model_name"].isin(["one_hot", "linreg"])]
    df = df[df["clf_type"] == "mlp"]

    return df.reset_index(drop=True)


raw_df = pd.read_csv(DATA_PATH)
df = preprocess_merged_csv(raw_df)

print(f"raw rows: {len(raw_df):,}")
print(f"preprocessed rows: {len(df):,}")
display(df.head())


In [ ]:
def available_features(sub_df, feature_columns=FEATURE_COLUMNS):
    return [col for col in feature_columns if col in sub_df.columns and sub_df[col].notna().any()]


def fdr_bh(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    adjusted = np.full(pvalues.shape, np.nan, dtype=float)
    valid = np.isfinite(pvalues)
    if valid.sum() == 0:
        return adjusted

    valid_pvalues = pvalues[valid]
    order = np.argsort(valid_pvalues)
    ranked = valid_pvalues[order]
    n = len(ranked)

    ranked_adjusted = np.empty(n, dtype=float)
    running_min = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        running_min = min(running_min, ranked[i] * n / rank, 1.0)
        ranked_adjusted[i] = running_min

    restored = np.empty(n, dtype=float)
    restored[order] = ranked_adjusted
    adjusted[valid] = restored
    return adjusted


def add_fdr_by(df, p_col="P_value", group_cols=("analysis",)):
    df = df.copy()
    df["P_value_fdr_bh"] = np.nan
    for _, idx in df.groupby(list(group_cols), dropna=False).groups.items():
        df.loc[idx, "P_value_fdr_bh"] = fdr_bh(df.loc[idx, p_col].to_numpy())
    return df


## Statistical tests

Train/test logic here means:

- `same_train_mutations_compare_models`: within each dataset and train-mutation count, compare `model_name` groups.
- `same_test_mutations_compare_models`: within each dataset and test-mutation count, compare `model_name` groups.
- `same_test_mutations_compare_train_mutations`: within each dataset and test-mutation count, compare `train_mutations` groups.

The plots below use ANOVA p-values. Kruskal-Wallis results are kept in the output tables for inspection.


In [ ]:
def run_group_tests(
    df,
    *,
    analysis,
    comparison_type,
    compare_col,
    fixed_cols,
    feature_columns=FEATURE_COLUMNS,
    min_groups=MIN_GROUPS,
    min_points_per_group=MIN_POINTS_PER_GROUP,
):
    rows = []

    for fixed_values, sub_df in df.groupby(fixed_cols, dropna=False):
        if not isinstance(fixed_values, tuple):
            fixed_values = (fixed_values,)
        fixed = dict(zip(fixed_cols, fixed_values))

        for feature in available_features(sub_df, feature_columns):
            groups = []
            group_names = []
            for group_name, group_df in sub_df.groupby(compare_col, dropna=False):
                values = group_df[feature].dropna().to_numpy(dtype=float)
                if len(values) >= min_points_per_group:
                    groups.append(values)
                    group_names.append(group_name)

            row = {
                "analysis": analysis,
                "comparison_type": comparison_type,
                "feature": feature,
                "compare_col": compare_col,
                "fixed_cols": ",".join(fixed_cols),
                "n_groups": len(groups),
                "n_points": int(sum(len(g) for g in groups)),
                **fixed,
            }

            if len(groups) >= min_groups:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", stats.ConstantInputWarning)
                    f_stat, p_value = stats.f_oneway(*groups)
                try:
                    k_stat, k_p_value = stats.kruskal(*groups)
                except ValueError:
                    k_stat, k_p_value = np.nan, np.nan
                row.update({
                    "test_type": "ANOVA",
                    "F_value": f_stat,
                    "P_value": p_value,
                    "Kruskal_H": k_stat,
                    "Kruskal_P": k_p_value,
                    "groups": ",".join(map(str, group_names)),
                })
            else:
                row.update({
                    "test_type": "insufficient_groups",
                    "F_value": np.nan,
                    "P_value": np.nan,
                    "Kruskal_H": np.nan,
                    "Kruskal_P": np.nan,
                    "groups": ",".join(map(str, group_names)),
                })

            rows.append(row)

    return pd.DataFrame(rows)


In [ ]:
model_pvalue_df = pd.concat(
    [
        run_group_tests(
            df,
            analysis="same_train_mutations_compare_models",
            comparison_type="train",
            compare_col="model_name",
            fixed_cols=["dataset", "train_mutations"],
        ),
        run_group_tests(
            df,
            analysis="same_test_mutations_compare_models",
            comparison_type="test",
            compare_col="model_name",
            fixed_cols=["dataset", "test_mutations"],
        ),
    ],
    ignore_index=True,
)
model_pvalue_df = add_fdr_by(model_pvalue_df, group_cols=("comparison_type",))

train_mutation_pvalue_df = run_group_tests(
    df,
    analysis="same_test_mutations_compare_train_mutations",
    comparison_type="test",
    compare_col="train_mutations",
    fixed_cols=["dataset", "test_mutations"],
)
train_mutation_pvalue_df = add_fdr_by(train_mutation_pvalue_df, group_cols=("analysis",))

if SAVE_TABLES:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    model_pvalue_df.to_csv(OUT_DIR / "merged_model_comparison_pvalues.csv", index=False)
    train_mutation_pvalue_df.to_csv(OUT_DIR / "merged_train_mutation_comparison_pvalues.csv", index=False)

print("model-comparison tests:", model_pvalue_df["P_value"].notna().sum(), "valid p-values")
print("train-mutation tests:", train_mutation_pvalue_df["P_value"].notna().sum(), "valid p-values")
display(model_pvalue_df.head())
display(train_mutation_pvalue_df.head())


## P-value distributions

In [ ]:
def plot_pvalue_cdf(df, p_col, *, title, panel_col=None, output_name=None):
    if panel_col is None:
        panels = [("All", df)]
    else:
        panels = [("All", df)] + [
            (str(value), df[df[panel_col] == value])
            for value in sorted(df[panel_col].dropna().unique())
        ]

    fig, axes = plt.subplots(1, len(panels), figsize=(3.0 * len(panels), 2.8), sharey=True)
    if len(panels) == 1:
        axes = [axes]

    for ax, (label, panel_df) in zip(axes, panels):
        pvalues = panel_df[p_col].dropna().sort_values().to_numpy()
        if len(pvalues):
            cumulative = 1 - np.arange(1, len(pvalues) + 1) / len(pvalues)
            significant = 100 * np.mean(pvalues < 0.05)
            ax.step(pvalues[::-1], cumulative[::-1], where="post", lw=2)
            ax.set_title(f"{label}\n{significant:.1f}% < 0.05")
        else:
            ax.set_title(f"{label}\nno p-values")

        ax.axvline(0.05, color="black", linestyle="--", lw=1)
        ax.set_xlim(1.05, -0.02)
        ax.set_ylim(-0.02, 1.02)
        ax.set_xlabel(p_col)
        ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.5)
        ax.spines["right"].set_visible(False)
        ax.spines["top"].set_visible(False)

    axes[0].set_ylabel("Fraction >= p-value")
    fig.suptitle(title, y=1.04)
    fig.tight_layout()

    if SAVE_FIGURES and output_name is not None:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUT_DIR / output_name, bbox_inches="tight")

    return fig


In [ ]:
plot_pvalue_cdf(
    model_pvalue_df,
    "P_value",
    title="Model comparison p-values, uncorrected",
    panel_col="comparison_type",
    output_name="merged_model_pvalue_distribution_uncorrected.svg",
)
plot_pvalue_cdf(
    model_pvalue_df,
    "P_value_fdr_bh",
    title="Model comparison p-values, BH-FDR corrected",
    panel_col="comparison_type",
    output_name="merged_model_pvalue_distribution_fdr.svg",
)
plt.show()


In [ ]:
plot_pvalue_cdf(
    train_mutation_pvalue_df,
    "P_value",
    title="Same dataset/test set, different train mutations: uncorrected",
    output_name="merged_train_mutation_pvalue_distribution_uncorrected.svg",
)
plot_pvalue_cdf(
    train_mutation_pvalue_df,
    "P_value_fdr_bh",
    title="Same dataset/test set, different train mutations: BH-FDR corrected",
    output_name="merged_train_mutation_pvalue_distribution_fdr.svg",
)
plt.show()


## Variance explained

By default this calculates a lean one-way eta-squared value (`SS_factor / SS_total`) for each dataset, feature, and factor. To read an existing table instead, set `VAR_EXPLAINED_PATH` in the config cell.


In [ ]:
def read_table(path):
    path = Path(path).expanduser()
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix in {".tsv", ".tab"}:
        return pd.read_csv(path, sep="	")
    return pd.read_csv(path)


def one_way_variance_explained(sub_df, feature, factor):
    values = sub_df[[feature, factor]].dropna()
    if values[feature].nunique() <= 1 or values[factor].nunique() <= 1:
        return np.nan

    y = values[feature].to_numpy(dtype=float)
    overall_mean = y.mean()
    total_ss = np.sum((y - overall_mean) ** 2)
    if total_ss == 0:
        return np.nan

    grouped = values.groupby(factor)[feature].agg(["mean", "count"])
    factor_ss = np.sum(grouped["count"] * (grouped["mean"] - overall_mean) ** 2)
    return factor_ss / total_ss


def calculate_variance_explained(df, factors=("model_name", "train_mutations", "test_mutations")):
    rows = []
    for dataset, dataset_df in df.groupby("dataset"):
        for feature in available_features(dataset_df):
            for factor in factors:
                rows.append({
                    "dataset": dataset,
                    "feature": feature,
                    "factor": factor,
                    "var_explained": one_way_variance_explained(dataset_df, feature, factor),
                })
    return pd.DataFrame(rows)


def load_or_calculate_variance_explained():
    if VAR_EXPLAINED_PATH is not None:
        var_df = read_table(VAR_EXPLAINED_PATH)
    elif CALCULATE_VARIANCE_EXPLAINED_IF_MISSING:
        var_df = calculate_variance_explained(df)
    else:
        return pd.DataFrame()

    var_df = var_df.rename(columns={"var": "feature"}).copy()
    required = {"dataset", "feature", "factor", "var_explained"}
    missing = required - set(var_df.columns)
    if missing:
        raise ValueError(f"variance-explained table is missing columns: {sorted(missing)}")
    return var_df


var_explained_df = load_or_calculate_variance_explained()

if SAVE_TABLES and len(var_explained_df):
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    var_explained_df.to_csv(OUT_DIR / "merged_variance_explained.csv", index=False)

display(var_explained_df)


In [ ]:
def plot_variance_explained(var_df, output_name="merged_variance_explained.svg"):
    if var_df.empty:
        print("No variance-explained data to plot.")
        return None

    plot_df = var_df.rename(columns={"var": "feature"}).copy()
    plot_df = plot_df.dropna(subset=["var_explained"])
    if plot_df.empty:
        print("Variance-explained data only contains NaN values.")
        return None

    if plot_df["var_explained"].max() <= 1.5:
        plot_df["var_explained"] = 100 * plot_df["var_explained"]

    plot_df["dataset_feature"] = plot_df["dataset"].astype(str) + "\n" + plot_df["feature"].astype(str)
    summary = (
        plot_df.groupby(["dataset_feature", "factor"], as_index=False)["var_explained"]
        .mean()
    )
    pivot = summary.pivot(index="dataset_feature", columns="factor", values="var_explained").fillna(0)

    ax = pivot.plot(kind="bar", figsize=(max(6, 0.55 * len(pivot)), 3.2), width=0.82)
    ax.set_ylabel("Variance explained (%)")
    ax.set_xlabel("")
    ax.grid(True, axis="y", linestyle="--", linewidth=0.4, alpha=0.5)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.legend(title="factor", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    if SAVE_FIGURES:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        ax.figure.savefig(OUT_DIR / output_name, bbox_inches="tight")

    return ax.figure


plot_variance_explained(var_explained_df)
plt.show()
